### RAG Application With Conversational Memory

following program is chatbot with coversational memory. We will load my IT courses website and have chat with it

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain


from langsmith import traceable, Client,trace
client = Client()
api_key = os.getenv("GOOGLE_API_KEY")
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
langsmith_tracing = os.getenv("LANGCHAIN_TRACING_V2")
os.environ["GOOGLE_API_KEY"] = str(api_key)
os.environ["LANGCHAIN_API_KEY"] = str(langchain_api_key)
os.environ["LANGCHAIN_TRACING_V2"] = str(langsmith_tracing)

<B>Load Page

In [2]:
loader = WebBaseLoader("https://www.bahria.edu.pk/Home/AcademicRoadmapDetails?roadmapId=46")

docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter()
document = text_splitter.split_documents(docs)

In [3]:
cnt =0
for doc in document:
    cnt = cnt+1
    print(cnt)

1
2
3


<b>Set the Model

In [4]:

llm = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), temperature=0, model="gemini-3.5-flash")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [5]:
vector = FAISS.from_documents(document, embeddings)
retriever = vector.as_retriever()

<b> Build Prompt

In [6]:
prompt = ChatPromptTemplate([
    MessagesPlaceholder(variable_name="chat_history"),
    ("user","{input}"),
    ("user","Given the above conversation, generate a search query to look up in order to get information relevant to the conversation")
])

In [7]:
# if there is not chat_history then the inut is just passed directly to the retriever. If there is chat history then the prompt and LLM will be used to generate a search query. Thet search query will then passed to the retriever
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)


In [8]:
sample_answer = """Some context about page is:
1. It is a BS-IT courses list of Bahria university.
2. This page cover elective courses as well
"""

In [9]:
chat_history = [HumanMessage(content = "What is a context of a page"),
                AIMessage(content = sample_answer)]

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system","Answer the user questions based on the below context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user","{input}")
])

In [ ]:
document_chain = create_stuff_documents_chain(llm , prompt)
retriever_chain = create_retrieval_chain(retriever_chain, document_chain)

In [15]:
output = retriever_chain.invoke({
    "chat_history": chat_history,
    "input":"is this contain Introduction to AI?"
})

In [19]:
print(output['answer'])

Based on the provided context, there is no course specifically titled "Introduction to AI." 

However, the curriculum does contain the following related courses:
* **Artificial Intelligence** (Course Code: CSC 411) and **Artificial Intelligence Lab** (Course Code: AIL 201) in **Semester 6**.
* **Introduction to Machine Learning** (Course Code: CSC 413) and its lab in the elective courses list.


In [17]:
chat_history.append(HumanMessage(content="is this contain Introduction to AI?"))
chat_history.append(AIMessage(content=output["answer"]))

In [18]:
output_turn_3 = retriever_chain.invoke({
    "chat_history": chat_history,
    "input": "In which semester i can choose it?"
})
print("Turn 3 Response:", output_turn_3["answer"])

Turn 3 Response: Based on the curriculum:

* **Artificial Intelligence (CSC 411):** This is a core (compulsory) course that is scheduled for **Semester 6**.
* **Introduction to Machine Learning (CSC 413):** This is an elective course. Because its prerequisite is CSC 411 (Artificial Intelligence, which you study in Semester 6), you can choose to take this elective in **Semester 7** or **Semester 8**.
